# Review a saved diffusion run

This notebook loads a saved run, rebuilds its model from the run's copied `config.toml`, and restores the checkpoint weights. It shows a forward noising trajectory for validation images and a reverse denoising trajectory starting from Gaussian noise.

The paths below assume the notebook is run with the repository's `notebooks/` directory as the working directory.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import torch
from torchvision.utils import make_grid

from radio_diffusion.experiment import (
    build_data_module,
    load_config,
    load_experiment_checkpoint,
)

## Choose a saved run

The picker scans `outputs/` recursively, so it can find runs from different configs and output folders. It shows the total number of checkpointed runs and labels the five newest with their experiment name and model settings. Set `run_index` to choose one; index 0 is the newest.

In [ ]:
output_root = Path("../outputs")
run_dirs = sorted(
    {
        config_file.parent
        for config_file in output_root.rglob("config.toml")
        if (config_file.parent / "checkpoints" / "best.ckpt").is_file()
        or (config_file.parent / "checkpoints" / "last.ckpt").is_file()
    },
    key=lambda path: max(
        checkpoint.stat().st_mtime
        for checkpoint in (
            path / "checkpoints" / "best.ckpt",
            path / "checkpoints" / "last.ckpt",
        )
        if checkpoint.is_file()
    ),
    reverse=True,
)
if not run_dirs:
    raise FileNotFoundError(f"No checkpointed runs found under {output_root}")

run_configs = [load_config(path / "config.toml") for path in run_dirs]
experiment_names = {config["run"]["name"] for config in run_configs}
print(f"Available checkpointed runs: {len(run_dirs)}")
print(f"Distinct experiment names: {len(experiment_names)}")
print("Newest runs:")
for index, (path, run_config) in enumerate(zip(run_dirs[:5], run_configs[:5])):
    model_config = run_config["model"]
    model_summary = f"{model_config.get('type', 'model')} channels={model_config.get('channels', 'default')}"
    if model_config.get("attention_levels"):
        model_summary += f" attention={model_config['attention_levels']}"
    print(f"{index}: {run_config['run']['name']} | {model_summary} | {path.name}")
if len(run_dirs) > 5:
    print(f"... and {len(run_dirs) - 5} older runs")

# Choose one of the listed runs. Index 0 is the newest.
run_index = 0
run_dir = run_dirs[run_index]
print("\nSelected run:", run_dir)
print("Saved config:", run_dir / "config.toml")

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
config = load_config(run_dir / "config.toml")
module = load_experiment_checkpoint(run_dir, map_location=device)

print("Loaded checkpoint for:", config["run"]["name"])
print("Device:", device)

## Display a saved training sample grid

The training callback writes a raw tensor and a PNG every configured number of epochs. This cell displays the latest tensor snapshot, if the run lasted long enough to produce one.

In [ ]:
sample_files = sorted((run_dir / "samples").glob("*.pt"))
if sample_files:
    saved_samples = torch.load(sample_files[-1], map_location="cpu")
    display_images = ((saved_samples + 1) / 2).clamp(0, 1)
    grid = make_grid(display_images, nrow=4)
    fig, ax = plt.subplots(figsize=(10, 8))
    ax.imshow(grid[0].numpy(), cmap="inferno", vmin=0, vmax=1)
    ax.set_title(sample_files[-1].name)
    ax.axis("off")
    plt.show()
else:
    print("No periodic sample snapshot was saved for this run.")

## Load validation images and define a trajectory viewer

In [ ]:
# Keep notebook data loading lightweight even if training used many workers.
config["data"]["num_workers"] = 0
data = build_data_module(config)
data.prepare_data()
data.setup("validate")
clean_images = next(iter(data.val_dataloader()))[:4].to(device)

def show_trajectory(states, labels, title):
    """Plot states shaped [batch, snapshots, channels, height, width]."""
    states = states.detach().cpu()
    batch_size, snapshot_count = states.shape[:2]
    fig, axes = plt.subplots(
        batch_size,
        snapshot_count,
        figsize=(2.6 * snapshot_count, 2.6 * batch_size),
        squeeze=False,
    )

    for image_index in range(batch_size):
        for snapshot_index, label in enumerate(labels):
            image = ((states[image_index, snapshot_index, 0] + 1) / 2).clamp(0, 1)
            ax = axes[image_index, snapshot_index]
            ax.imshow(image, cmap="inferno", vmin=0, vmax=1)
            ax.axis("off")
            if image_index == 0:
                ax.set_title(label)
        axes[image_index, 0].set_ylabel(f"Image {image_index + 1}")

    fig.suptitle(title)
    fig.tight_layout()
    plt.show()

## Forward process: add noise to clean images

Each row shows one validation image at increasing noise levels. Reusing the same noise tensor across the selected timesteps makes the progression easier to compare.

In [ ]:
torch.manual_seed(config["run"]["seed"])
noise = torch.randn_like(clean_images)
timesteps = torch.linspace(
    0,
    module.diffusion.num_steps - 1,
    6,
    dtype=torch.long,
    device=device,
)
noisy_trajectory, timesteps = module.diffusion.forward_trajectory(
    clean_images,
    timesteps,
    noise=noise,
)
labels = ["clean"] + [f"t={t.item()}" for t in timesteps]
show_trajectory(noisy_trajectory, labels, "Forward diffusion")

## Reverse process: denoise Gaussian noise

This follows the same reverse steps used by DDPM sampling. The first column is the initial Gaussian noise; later columns show selected intermediate states, ending at the generated image. The seed fixes both the starting noise and the random noise used by reverse steps.

In [ ]:
n_samples = min(4, len(clean_images))
sample_shape = (n_samples, *clean_images.shape[1:])
num_steps = module.diffusion.num_steps
selected_timesteps = set(
    torch.linspace(num_steps - 1, 0, 6, dtype=torch.long).tolist()
)

torch.manual_seed(config["run"]["seed"])
generated = torch.randn(sample_shape, device=device)
reverse_states = [generated.detach().cpu()]
reverse_labels = ["initial noise"]

with torch.inference_mode():
    for timestep in reversed(range(num_steps)):
        batch_timesteps = torch.full(
            (n_samples,),
            timestep,
            dtype=torch.long,
            device=device,
        )
        generated = module.diffusion.reverse_step(
            module.model,
            generated,
            batch_timesteps,
        )
        if timestep in selected_timesteps:
            reverse_states.append(generated.detach().cpu())
            reverse_labels.append(f"after t={timestep}")

reverse_trajectory = torch.stack(reverse_states, dim=1)
show_trajectory(reverse_trajectory, reverse_labels, "Reverse denoising")
print("Final generated batch shape:", tuple(generated.shape))